# Creating a Simple Agent with Tracing

In [1]:
import dotenv
import os

from openai import OpenAI

dotenv.load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    print(
        """Error: OPENAI_API_KEY environment variable not set. Please copy the .env.template file as .env and fill it in.
    
    You can execute these commands in the terminal to get started:
    cp .env.template .env
    code .env
    """
    )

# Test OpenAI Access
print(
    OpenAI()
    .responses.create(
        model=os.environ["OPENAI_DEFAULT_MODEL"], input="Say: We are up and running!"
    )
    .output_text
)

We are up and running!


In [ ]:
from agents import Agent, Runner, trace
from openai.types.responses import ResponseTextDeltaEvent

0.3.0
['Agent', 'AgentBase', 'AgentHooks', 'AgentOutputSchema', 'AgentOutputSchemaBase', 'AgentSpanData', 'AgentUpdatedStreamEvent', 'AgentsException', 'AsyncComputer', 'AsyncOpenAI', 'Button', 'CodeInterpreterTool', 'Computer', 'ComputerTool', 'CustomSpanData', 'DynamicPromptFunction', 'Environment', 'FileSearchTool', 'FunctionSpanData', 'FunctionTool', 'FunctionToolResult', 'GenerateDynamicPromptData', 'GenerationSpanData', 'GuardrailFunctionOutput', 'GuardrailSpanData', 'Handoff', 'HandoffCallItem', 'HandoffInputData', 'HandoffInputFilter', 'HandoffOutputItem', 'HandoffSpanData', 'HostedMCPTool', 'ImageGenerationTool', 'InputGuardrail', 'InputGuardrailResult', 'InputGuardrailTripwireTriggered', 'ItemHelpers', 'Literal', 'LocalShellCommandRequest', 'LocalShellExecutor', 'LocalShellTool', 'MCPListToolsSpanData', 'MCPToolApprovalFunction', 'MCPToolApprovalFunctionResult', 'MCPToolApprovalRequest', 'MaxTurnsExceeded', 'MessageOutputItem', 'Model', 'ModelBehaviorError', 'ModelProvider', 

Create a simple Nutrition Assistant Agent

In [9]:
nutrition_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful assistant giving out nutrition advice.
    You give concise answers.
"""
)


Let's execute the Agent:

In [10]:
with agents.trace("Simple Nutrition Agent"):
    result = await Runner.run(nutrition_agent, "How healthy are bananas?")

print(result)

RunResult:
- Last agent: Agent(name="Nutrition Assistant", ...)
- Final output (str):
    Bananas are a healthy, convenient fruit. Pros:
    - Good source of potassium, vitamin B6, vitamin C, and fiber
    - Low in fat and calories, with natural sugars for quick energy
    - Contain resistant starch when less ripe, which can aid digestion
    
    Considerations:
    - Sugar content and carbs add up if you’re watching glycol intake (e.g., for diabetes); portion matters (1 medium banana ≈ 105 calories)
    - People with kidney disease may need to limit potassium; check with a clinician
    
    Bottom line: they’re a nutritious, versatile fruit when eaten in moderation as part of a balanced diet.
- 2 new item(s)
- 1 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)


Streaming the answer to the screen, token by token

In [11]:
response_stream = Runner.run_streamed(nutrition_agent, "How healthy are bananas?")

async for event in response_stream.stream_events():
    if event.type == "raw_response_event" and isinstance(
        event.data, ResponseTextDeltaEvent
    ):
        print(event.data.delta, end="", flush=True)

Bananas are a healthy, nutrient-dense fruit when eaten in moderation.

Key benefits:
- Good source of potassium (supports heart and blood pressure) and vitamin B6.
- Provides vitamin C, fiber, and some magnesium.
- Convenient, portable energy from natural sugars and carbs.

Things to note:
- Moderate portions (1 medium banana ~ 105 calories) fit most diets.
- Higher sugar content as they ripen; choose based on your blood sugar needs.
- Diabetics should monitor portion size and total carbohydrate intake.

Tips:
- Pair with protein or fat (e.g., peanut butter, yogurt) to curb blood sugar spikes.
- Use green/unripe bananas for more resistant starch (lower sugar; slower digestion).

_Good Job!_